# Forecasting `Ziekteverzuimpercentage` — Seasonal Baseline vs LinearRegression

A minimal end-to-end pipeline that forecasts the national quarterly sickness-absence rate (`Ziekteverzuimpercentage_1`) **3 quarters ahead** and compares a 3-year seasonal moving-average baseline against a LinearRegression fitted on 61 macro-labour features.

## Approach

**Data.** A single quarterly time series (1996 Q1 onward, 120 observations) of the Dutch national sickness-absence rate, joined with 61 macro-labour features from the gold-layer feature store (`master_data_ml_preprocessed`).

**Forecast horizon.** Three quarters ahead. To make this an honest forecasting setup — rather than a contemporaneous mapping — the regression is trained on **features lagged by 3 quarters**: at time `t`, the model only sees information that was already available at time `t-3`.

**Train / test split.** Chronological, *not* random — random splits leak future information into the training set. The last **12 quarters (3 years)** are held out as the test set, leaving 108 quarters for training.

**Models.**
- **A — Seasonal moving-average baseline:** `ŷ_t = mean(y_{t-4}, y_{t-8}, y_{t-12})` (same quarter, averaged over the previous three years). A simple baseline that smooths over year-to-year noise while preserving seasonality.
- **B — Linear Regression:** `sklearn.linear_model.LinearRegression` on the 61 lagged features. No scaling, no regularisation, no feature selection — all deliberately deferred to later iterations.

**Evaluation.** MAE, RMSE and R² on the 12-quarter hold-out, plus a visual actual-vs-predicted plot. The interesting question is not the absolute error but whether **Model B beats the seasonal baseline (A)** — if it doesn't, the features (or the linear specification) aren't pulling their weight.

## 1 · Environment setup

Imports the project's data, plotting and modelling stack, wires the repo root into `sys.path` so that `from src.*` works inside the notebook, and pulls the two project-wide constants we will rely on — the path to the gold SQLite database and the name of the target column. Also declares `STRUCTURAL_COLS`: the set of columns that act as identifiers / metadata and must never be fed to a model as features.

In [49]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold

# Make src/ importable from notebook context
PROJECT_ROOT = Path().resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DIR_DB_GOLD, ML_TARGET_COLUMN

# ── Structural columns — never treated as ML features ──────────────────────
STRUCTURAL_COLS = {"silver_id", "quarter", "year", "period_enddate", "BedrijfstakkenBranchesSBI2008"}

print(f"Project root : {PROJECT_ROOT}")
print(f"Gold DB      : {DIR_DB_GOLD}")
print(f"Target column: {ML_TARGET_COLUMN}")

Project root : C:\Users\User\Documents\GitHub\eaisi-uwv
Gold DB      : C:\Users\User\Documents\GitHub\eaisi-uwv\data\3_gold\gold_data.db
Target column: Ziekteverzuimpercentage_1


## 2 · Extract the gold-layer feature store

Instantiates `DataExtractor` against the `master_data_ml_preprocessed` table and pulls a flat DataFrame in **discovery mode** (`feature_groups=None`) — i.e. *all* columns rather than a curated subset. The result `df_raw` has one row per quarter at the national / all-industry level, with the target alongside every candidate feature.

In [50]:
from src.ml_engineering.ml_1_data_extraction import DataExtractor

extractor = DataExtractor(db_path=DIR_DB_GOLD, table_name="master_data_ml_preprocessed")
df_raw = extractor.extract(target_column=ML_TARGET_COLUMN, feature_groups=None)  # discovery: all columns

print(f"Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
df_raw.head(10)

⚙️ SBI mode=all-industry (BedrijfskenmerkenSBI2008_T001081) | 120 quarterly rows
✅ Extraction complete | mode=discovery | sbi=all-industry | features=61 | rows=120


Shape: 120 rows × 65 columns


,period_enddate,year,quarter,Ziekteverzuimpercentage_1,trend_index,Banen_2_A045285_3000,Banen_2_A045285_4000,Banen_2_A045286_3000,Banen_2_A045286_4000,Banen_7_A045285_3000,...,GewerkteUren_5_A045285,GewerkteUren_5_A045286,WerkzamePersonenSeizoengecorrigeerd_2_A045285,WerkzamePersonenSeizoengecorrigeerd_2_A045286,WerkzamePersonenSeizoengecorrigeerd_9_A045285,WerkzamePersonenSeizoengecorrigeerd_9_A045286,WerkzamePersonen_1_A045285,WerkzamePersonen_1_A045286,WerkzamePersonen_7_A045285,WerkzamePersonen_7_A045286
0,1996-03-31 00:00:00.000000,1996.0,1.0,5.5,1.0,4295.0,3895.0,1098.0,1088.5,4296.0,...,216.866667,44.733333,589.733333,102.600000,0.800000,-0.066667,582.933333,102.000000,13.200000,0.733333
1,1996-06-30 00:00:00.000000,1996.0,2.0,4.6,2.0,4295.0,3895.0,1098.0,1088.5,4296.0,...,202.733333,46.600000,594.066667,102.600000,4.333333,0.066667,596.466667,102.733333,15.200000,0.266667
2,1996-09-30 00:00:00.000000,1996.0,3.0,4.0,3.0,4295.0,3895.0,1098.0,1088.5,4296.0,...,198.866667,43.866667,597.866667,102.733333,3.800000,0.266667,604.133333,102.866667,14.800000,0.400000
3,1996-12-31 00:00:00.000000,1996.0,4.0,4.7,4.0,4295.0,3895.0,1098.0,1088.5,4296.0,...,236.600000,48.133333,602.333333,103.600000,4.533333,0.800000,600.200000,103.666667,12.866667,0.666667
4,1997-03-31 00:00:00.000000,1997.0,1.0,4.9,5.0,4295.0,3895.0,1098.0,1088.5,4296.0,...,220.466667,45.133333,607.200000,104.800000,4.866667,1.066667,600.200000,104.600000,17.333333,2.600000
5,1997-06-30 00:00:00.000000,1997.0,2.0,4.5,6.0,4295.0,3895.0,1098.0,1088.5,4296.0,...,214.800000,47.400000,612.466667,105.466667,5.266667,0.800000,614.933333,105.733333,18.533333,2.800000
6,1997-09-30 00:00:00.000000,1997.0,3.0,4.1,7.0,4295.0,3895.0,1098.0,1088.5,4296.0,...,204.466667,44.466667,618.533333,105.666667,6.333333,0.066667,625.333333,105.400000,21.133333,2.800000
7,1997-12-31 00:00:00.000000,1997.0,4.0,4.9,8.0,4295.0,3895.0,1098.0,1088.5,4296.0,...,235.000000,48.666667,623.466667,104.933333,5.000000,-0.600000,621.000000,105.800000,20.733333,2.133333
8,1998-03-31 00:00:00.000000,1998.0,1.0,5.2,9.0,4295.0,3895.0,1098.0,1088.5,4296.0,...,230.200000,44.866667,627.933333,104.133333,4.600000,-0.866667,620.866667,103.133333,20.533333,-1.400000
9,1998-06-30 00:00:00.000000,1998.0,2.0,4.9,10.0,4295.0,3895.0,1098.0,1088.5,4296.0,...,216.733333,47.000000,633.533333,103.333333,5.533333,-0.733333,636.733333,103.466667,21.600000,-2.066667


## 3 · Column bookkeeping

Sorts every column of `df_raw` into one of four buckets — `structural`, `target`, `sector_filter (OHE)`, `feature` — so the downstream cells can cleanly refer to `feature_cols` without leaking metadata or the target into the feature matrix. The summary table also confirms how many features survived after exclusions.

In [40]:
SBI_FILTER_PREFIX = "BedrijfskenmerkenSBI2008_"

structural    = [c for c in df_raw.columns if c in STRUCTURAL_COLS]
target_col    = [ML_TARGET_COLUMN]
sector_filter = [c for c in df_raw.columns if c.startswith(SBI_FILTER_PREFIX)]

# Feature columns: everything else — exclude structural, target, and sector-filter OHE
feature_cols  = [
    c for c in df_raw.columns
    if c not in STRUCTURAL_COLS
    and c != ML_TARGET_COLUMN
    and not c.startswith(SBI_FILTER_PREFIX)
]

summary = pd.DataFrame({
    "Bucket":  ["structural", "target", "sector_filter (OHE)", "feature"],
    "Count":   [len(structural), len(target_col), len(sector_filter), len(feature_cols)],
    "Examples": [
        ", ".join(structural[:3]),
        ML_TARGET_COLUMN,
        ", ".join(sector_filter[:3]),
        ", ".join(feature_cols[:3]),
    ],
})
print(summary.to_string(index=False))
print(f"\nTotal ML features (excl. structural + target + sector_filter): {len(feature_cols)}")

example_col = sector_filter[0] if sector_filter else "BedrijfskenmerkenSBI2008_XXX"
print(f"\nSector filter columns ({len(sector_filter)}) — use to restrict analysis to one SBI sector:")
print(f"  e.g. df_raw[df_raw[{example_col!r}] == 1]")

             Bucket  Count                                                Examples
         structural      3                           period_enddate, year, quarter
             target      1                               Ziekteverzuimpercentage_1
sector_filter (OHE)      0                                                        
            feature     61 trend_index, Banen_2_A045285_3000, Banen_2_A045285_4000

Total ML features (excl. structural + target + sector_filter): 61

Sector filter columns (0) — use to restrict analysis to one SBI sector:
  e.g. df_raw[df_raw['BedrijfskenmerkenSBI2008_XXX'] == 1]


## 4 · Modelling imports

The few extra `sklearn` pieces needed for the basic pipeline: `LinearRegression` for the model itself and three metric helpers (`mean_absolute_error`, `mean_squared_error`, `r2_score`) for evaluation.

In [41]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 5 · Build a time-ordered DataFrame

Copies `df_raw`, converts `period_enddate` to a real `pandas.Timestamp` column, sorts the rows chronologically, and promotes the date column to the **DatetimeIndex** of `df`. From here on `df` is the canonical, time-ordered version of the data — needed for honest chronological splits and time-axis plots.

In [42]:
# Sort chronologically and set a proper DatetimeIndex (needed for time-aware splits and plots).
df = df_raw.copy()
df["period_enddate"] = pd.to_datetime(df["period_enddate"])
df = df.sort_values("period_enddate").set_index("period_enddate")

print(f"Rows       : {len(df)}")
print(f"Date range : {df.index.min().date()} → {df.index.max().date()}")

Rows       : 120
Date range : 1996-03-31 → 2025-12-31


## 6 · Chronological train / test split

Holds out the **last 12 quarters (3 years)** as the test set and uses everything before that for training. The split is purely positional (`df.iloc[:-TEST_SIZE]` / `df.iloc[-TEST_SIZE:]`) because the index is already time-ordered — no shuffling, no random seed. The printed date ranges are a quick sanity check that the split landed where expected.

In [43]:
# Chronological hold-out: last 12 quarters = 3 years = 4 non-overlapping 3Q forecasts.
TEST_SIZE = 12

train = df.iloc[:-TEST_SIZE]
test  = df.iloc[-TEST_SIZE:]

X_train, X_test = train[feature_cols], test[feature_cols]
y_train, y_test = train[ML_TARGET_COLUMN], test[ML_TARGET_COLUMN]

print(f"Train: {X_train.shape[0]} rows × {X_train.shape[1]} features  ({train.index.min().date()} → {train.index.max().date()})")
print(f"Test : {X_test.shape[0]} rows × {X_test.shape[1]} features  ({test.index.min().date()} → {test.index.max().date()})")

Train: 108 rows × 61 features  (1996-03-31 → 2022-12-31)
Test : 12 rows × 61 features  (2023-03-31 → 2025-12-31)


## 7 · Model A — Seasonal moving-average baseline

A simple seasonal baseline: predict each test quarter as the **mean of the same quarter over the previous three years** — `ŷ_t = mean(y_{t-4}, y_{t-8}, y_{t-12})`. No fitting, no features. Averaging across three years smooths over single-year shocks (e.g. the COVID-era spike) while still preserving the quarterly seasonal pattern.

In [44]:
# ── Model A: Seasonal moving-average baseline (3-year same-quarter mean) ──
# Prediction for quarter t = mean(y_{t-4}, y_{t-8}, y_{t-12}).
y_full = df[ML_TARGET_COLUMN]
LAGS_SMA = [4, 8, 12]
y_pred_sma = (
    pd.concat([y_full.shift(k) for k in LAGS_SMA], axis=1)
      .mean(axis=1)
      .iloc[-TEST_SIZE:]
)

print(f"Seasonal-MA predictions for the last {TEST_SIZE} quarters:")
print(pd.DataFrame(
    {"actual": y_test.values, "sma_pred": y_pred_sma.round(3).values},
    index=test.index,
))

Seasonal-MA predictions for the last 12 quarters:
                actual  sma_pred
period_enddate                  
2023-03-31         5.7     5.433
2023-06-30         5.0     4.867
2023-09-30         4.8     4.667
2023-12-31         5.5     5.300
2024-03-31         5.5     5.600
2024-06-30         5.1     5.033
2024-09-30         4.9     4.800
2024-12-31         5.4     5.500
2025-03-31         5.8     5.833
2025-06-30         5.2     5.167
2025-09-30         5.1     4.900
2025-12-31         5.6     5.500


## 8 · Model B — LinearRegression with 3-quarter lagged features

Shifts every feature by 3 quarters so that the matrix `X_lagged_t` only contains information that was known at time `t-3`. The model therefore learns the relationship `y_t = f(X_{t-3})` directly — a **direct 3-step-ahead** forecasting setup, no recursive prediction needed. The first 3 rows (where the lagged features are NaN) are dropped before splitting and fitting.

In [ ]:
# ── Model B: LinearRegression with 3-quarter lagged features ──────────
# To forecast 3 quarters ahead, the model learns y_t = f(X_{t-3}).
LAG = 3
X_lagged = df[feature_cols].shift(LAG)

# Align: drop the first LAG rows where lagged features are NaN.
mask = X_lagged.notna().all(axis=1)
X_lagged_aligned = X_lagged.loc[mask]
y_aligned        = df.loc[mask, ML_TARGET_COLUMN]

# Same chronological split — last 12 quarters as test.
X_train_lag = X_lagged_aligned.iloc[:-TEST_SIZE]
X_test_lag  = X_lagged_aligned.iloc[-TEST_SIZE:]
y_train_lag = y_aligned.iloc[:-TEST_SIZE]
y_test_lag  = y_aligned.iloc[-TEST_SIZE:]

lr = LinearRegression()
lr.fit(X_train_lag, y_train_lag)
y_pred_lr = lr.predict(X_test_lag)

print(f"Lagged train shape: {X_train_lag.shape}")
print(f"Lagged test  shape: {X_test_lag.shape}")
print(f"First test prediction: {y_pred_lr[0]:.3f}  (actual: {y_test_lag.iloc[0]:.3f})")

## 9 · Hold-out metrics

Computes MAE, RMSE and R² for both models on the 12-quarter hold-out and prints them as a single comparison table. The interesting question: does the LinearRegression (B) beat the seasonal moving-average baseline (A)?

What each metric means:

**MAE** (Mean Absolute Error) — average absolute distance between prediction and truth, in the same units as the target (percentage points). MAE = 0.22 means "on average we miss the true sickness-absence rate by 0.22 pp."

**RMSE** (Root Mean Squared Error) — same units as the target, but squares errors before averaging, so it punishes big misses much harder than MAE. RMSE ≥ MAE always; a large gap signals a few large errors rather than uniformly small ones.

**R²** — share of the variance in the test target explained by the model, relative to just predicting the test mean. R² = 1 is perfect, R² = 0 means "no better than guessing the mean", negative R² means worse than guessing the mean.

In [ ]:
def score(y_true, y_pred):
    return {
        "MAE":  mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R²":   r2_score(y_true, y_pred),
    }

metrics = pd.DataFrame(
    [
        score(y_test, y_pred_sma),
        score(y_test, y_pred_lr),
    ],
    index=[
        "Seasonal MA (3yr same-quarter)",
        "LinearRegression (lag 3Q)",
    ],
).round(4)

print("Hold-out metrics (last 12 quarters):")
print(metrics.to_string())

## 10 · Actual vs predicted

A single plot of the 12-quarter hold-out: the true target in solid black, the seasonal moving-average prediction in dashed green, and the LinearRegression prediction in dashed blue. Visual inspection often reveals failure modes that the aggregate metrics hide — e.g. one model tracking the level but missing the swings, or systematically over- or under-shooting in a specific quarter.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(test.index, y_test.values,       label="Actual",                          color="black",      linewidth=2)
ax.plot(test.index, y_pred_sma.values,   label="Seasonal MA (3yr same-quarter)",  color="tab:green",  linestyle="--")
ax.plot(test.index, y_pred_lr,           label="LinearRegression (lag 3Q)",       color="tab:blue",   linestyle="--")

ax.set_title("Actual vs predicted on 12-quarter hold-out")
ax.set_xlabel("Quarter")
ax.set_ylabel(ML_TARGET_COLUMN)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()